## Setup and load data

This is the rebuilt final run using the stratified folds and cross-fitting for calibration. Code 15 is held out for the out-of-distribution test and excluded from the main cross-validation. Seeds are fixed so the run is reproducible.

In [1]:
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False

device="cuda" if torch.cuda.is_available() else "cpu"
print("device:",device)
print("gpu:",torch.cuda.get_device_name(0) if device=="cuda" else "none")

DATA="/kaggle/input/datasets/arpitjoshua/mt-trajnet-thesis-data/kaggle_upload"

with open(DATA+"/assembled_trajectories.pkl","rb") as fh:
    data=pickle.load(fh)
X_traj=data["X_traj"]
y_arr=data["y_arr"]
groups=data["groups"]

print("loaded:",len(X_traj),"batches | y",y_arr.shape,"| codes",len(np.unique(groups)))
print("any NaN:",any(np.isnan(a).any() for a in X_traj))

device: cuda
gpu: Tesla T4
loaded: 1005 batches | y (1005, 4) | codes 25
any NaN: False


## Data preparation

Stride-2 downsampling and a length cap of 6000, with normalisation computed from training data inside each fold.

In [2]:
STRIDE=2
MAXLEN=6000
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]

def prep_batch(traj_list,mean,std,maxlen=MAXLEN):
    out=[]
    for a in traj_list:
        a=a[::STRIDE]
        a=(a-mean)/std
        if len(a)<maxlen:
            pad=np.zeros((maxlen-len(a),a.shape[1]),dtype="float32")
            a=np.vstack([a,pad])
        else:
            a=a[:maxlen]
        out.append(a)
    arr=np.array(out,dtype="float32")
    return torch.tensor(arr).transpose(1,2)

covered=sum(1 for a in X_traj if len(a[::STRIDE])<=MAXLEN)
print("stride",STRIDE,"maxlen",MAXLEN)
print("fully covered:",covered,"of",len(X_traj),f"({100*covered/len(X_traj):.0f}%)")

stride 2 maxlen 6000
fully covered: 939 of 1005 (93%)


## Model: TCN encoder with evidential heads

The same architecture as before: a shared TCN encoder with four dilated residual blocks, and four evidential heads outputting the parameters of a Normal-Inverse-Gamma distribution. The evidential parameters are clamped to a safe range so the variance cannot diverge.

In [3]:
class TCNBlock(nn.Module):
    def __init__(self,in_ch,out_ch,dilation,kernel=3,dropout=0.1):
        super().__init__()
        pad=(kernel-1)*dilation
        self.conv1=nn.Conv1d(in_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.conv2=nn.Conv1d(out_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.relu=nn.ReLU()
        self.drop=nn.Dropout(dropout)
        self.pad=pad
        self.down=nn.Conv1d(in_ch,out_ch,1) if in_ch!=out_ch else None
    def forward(self,x):
        res=x if self.down is None else self.down(x)
        out=self.conv1(x)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        out=self.conv2(out)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        return self.relu(out+res)

class TCNEncoder(nn.Module):
    def __init__(self,in_ch=10,hidden=128,dropout=0.1):
        super().__init__()
        self.b1=TCNBlock(in_ch,hidden,1,dropout=dropout)
        self.b2=TCNBlock(hidden,hidden,2,dropout=dropout)
        self.b3=TCNBlock(hidden,hidden,4,dropout=dropout)
        self.b4=TCNBlock(hidden,hidden,8,dropout=dropout)
    def forward(self,x):
        x=self.b1(x);x=self.b2(x);x=self.b3(x);x=self.b4(x)
        return x.mean(dim=2)

class EvidentialHead(nn.Module):
    def __init__(self,in_dim,hidden=64,dropout=0.1):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hidden),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden,4))
    def forward(self,x):
        out=self.net(x)
        gamma=out[:,0]
        nu=F.softplus(out[:,1]).clamp(min=1e-2,max=1e3)
        alpha=F.softplus(out[:,2]).clamp(min=1e-2,max=1e3)+1.0
        beta=F.softplus(out[:,3]).clamp(min=1e-2,max=1e3)
        return gamma,nu,alpha,beta

def evidential_loss(y,gamma,nu,alpha,beta,lam=1.0):
    om=2*beta*(1+nu)
    nll=(0.5*torch.log(np.pi/nu)-alpha*torch.log(om)
         +(alpha+0.5)*torch.log(nu*(y-gamma)**2+om)
         +torch.lgamma(alpha)-torch.lgamma(alpha+0.5))
    reg=torch.abs(y-gamma)*(2*nu+alpha)
    return (nll+lam*reg).mean()

class MTTrajNet(nn.Module):
    def __init__(self,in_ch=10,hidden=128,n_targets=4,dropout=0.1):
        super().__init__()
        self.encoder=TCNEncoder(in_ch,hidden,dropout)
        self.heads=nn.ModuleList([EvidentialHead(hidden,dropout=dropout) for _ in range(n_targets)])
    def forward(self,x):
        z=self.encoder(x)
        outs=[h(z) for h in self.heads]
        gamma=torch.stack([o[0] for o in outs],dim=1)
        nu=torch.stack([o[1] for o in outs],dim=1)
        alpha=torch.stack([o[2] for o in outs],dim=1)
        beta=torch.stack([o[3] for o in outs],dim=1)
        return gamma,nu,alpha,beta

m=MTTrajNet().to(device)
t=torch.randn(3,10,500).to(device)
g,n,a,b=m(t)
var=b/(n*(a-1))
print("predictions:",g.shape,"| variance finite:",torch.isfinite(var).all().item())
print("parameters:",sum(p.numel() for p in m.parameters()))

predictions: torch.Size([3, 4]) | variance finite: True
parameters: 384400


## Stratified folds and the out-of-distribution hold-out

These are the fold assignments verified earlier. Code 15 is held out entirely for the out-of-distribution test. The four remaining high-hardness codes are distributed so the high-cluster batch counts per fold are 22, 6 and 6, rather than the 90, 6, 2 of the plain grouped split. I rebuild the batch indices for each fold from these code lists and confirm the high-cluster counts before training.

In [4]:
from collections import Counter

OOD_CODE=15
fold_codes=[
[1,2,4,6,7,9,13,18,25],
[10,11,17,19,22,24],
[3,5,8,12,14,16,20,21,23],
]
high_in_cv=[4,8,20,24]
n_splits=len(fold_codes)

counts=Counter(int(c) for c in groups)
ood_idx=np.where(groups==OOD_CODE)[0]

fold_test_idx=[]
for f in range(n_splits):
    idx=np.where(np.isin(groups,fold_codes[f]))[0]
    fold_test_idx.append(idx)

print("OOD held-out code:",OOD_CODE,f"({len(ood_idx)} batches)")
print()
allcv=[]
for f in range(n_splits):
    codes=fold_codes[f]
    hi=[c for c in codes if c in high_in_cv]
    hi_batches=sum(counts[c] for c in hi)
    print(f"fold {f+1}: {len(fold_test_idx[f])} batches | high-cluster {hi} -> {hi_batches} batches")
    allcv+=codes

print()
print("check: high-cluster per fold is 22/6/6:",
      [sum(counts[c] for c in fold_codes[f] if c in high_in_cv) for f in range(n_splits)]==[22,6,6])
print("check: code 15 excluded from CV:",OOD_CODE not in allcv)
print("check: no code repeated across folds:",len(allcv)==len(set(allcv)))
print("check: CV codes + OOD = 25:",len(set(allcv))+1==25)

OOD held-out code: 15 (64 batches)

fold 1: 314 batches | high-cluster [4] -> 22 batches
fold 2: 313 batches | high-cluster [24] -> 6 batches
fold 3: 314 batches | high-cluster [8, 20] -> 6 batches

check: high-cluster per fold is 22/6/6: True
check: code 15 excluded from CV: True
check: no code repeated across folds: True
check: CV codes + OOD = 25: True


## Cross-fitting training

Each fold trains on its full training set with no calibration partition removed, then predicts on its held-out test fold. Pooling these out-of-fold predictions gives a prediction for every batch from a model that never saw it. The calibration scale for each target is estimated from these pooled out-of-fold predictions, so no training data is sacrificed to calibration. A small validation split from the training products is still used for early stopping only.

In [5]:
from sklearn.metrics import mean_squared_error
from scipy.stats import norm
import time,warnings
warnings.filterwarnings("ignore")

def train_fold(tr,va,te,max_epochs=150,lr=5e-4,batch_size=16,patience=10,seed=SEED):
    xmean=np.vstack([X_traj[i][::STRIDE] for i in tr]).mean(axis=0)
    xstd=np.vstack([X_traj[i][::STRIDE] for i in tr]).std(axis=0)+1e-6
    Xtr=prep_batch([X_traj[i] for i in tr],xmean,xstd)
    Xva=prep_batch([X_traj[i] for i in va],xmean,xstd)
    Xte=prep_batch([X_traj[i] for i in te],xmean,xstd)
    ymean=y_arr[tr].mean(axis=0);ystd=y_arr[tr].std(axis=0)+1e-6
    ytr=torch.tensor((y_arr[tr]-ymean)/ystd,dtype=torch.float32)
    yva=torch.tensor((y_arr[va]-ymean)/ystd,dtype=torch.float32)

    torch.manual_seed(seed)
    model=MTTrajNet().to(device)
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    best=float("inf");best_state=None;wait=0
    for ep in range(max_epochs):
        model.train()
        perm=torch.randperm(len(Xtr))
        for i in range(0,len(Xtr),batch_size):
            idx=perm[i:i+batch_size]
            xb=Xtr[idx].to(device);yb=ytr[idx].to(device)
            opt.zero_grad()
            gg,nn_,aa,bb=model(xb)
            loss=sum(evidential_loss(yb[:,k],gg[:,k],nn_[:,k],aa[:,k],bb[:,k]) for k in range(4))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            vl=0
            for i in range(0,len(Xva),batch_size):
                xb=Xva[i:i+batch_size].to(device);yb=yva[i:i+batch_size].to(device)
                gg,nn_,aa,bb=model(xb)
                vl+=sum(evidential_loss(yb[:,k],gg[:,k],nn_[:,k],aa[:,k],bb[:,k]) for k in range(4)).item()
        if vl<best-1e-4:
            best=vl;best_state={kk:v.cpu().clone() for kk,v in model.state_dict().items()};wait=0
        else:
            wait+=1
            if wait>=patience:break
    model.load_state_dict(best_state);model.eval()

    gs=[];ss=[]
    with torch.no_grad():
        for i in range(0,len(Xte),batch_size):
            xb=Xte[i:i+batch_size].to(device)
            gg,nn_,aa,bb=model(xb)
            v=bb/(nn_*(aa-1))
            gs.append(gg.cpu().numpy());ss.append(torch.sqrt(v).cpu().numpy())
    pred=np.vstack(gs)*ystd+ymean
    std=np.vstack(ss)*ystd
    return pred,std

def run_crossfit():
    oof_pred=np.zeros((len(X_traj),4))
    oof_std=np.zeros((len(X_traj),4))
    oof_mask=np.zeros(len(X_traj),dtype=bool)
    fold_rmse=[]
    for f in range(n_splits):
        te=fold_test_idx[f]
        trv=np.concatenate([fold_test_idx[j] for j in range(n_splits) if j!=f])
        g=groups[trv];uniq=np.unique(g)
        rng=np.random.RandomState(SEED+f)
        val_codes=rng.choice(uniq,size=max(1,len(uniq)//8),replace=False)
        va=trv[np.isin(g,val_codes)]
        tr=trv[~np.isin(g,val_codes)]
        pred,std=train_fold(tr,va,te,seed=SEED+f)
        oof_pred[te]=pred;oof_std[te]=std;oof_mask[te]=True
        rmse=[np.sqrt(mean_squared_error(y_arr[te,k],pred[:,k])) for k in range(4)]
        fold_rmse.append(rmse)
        print(f"fold {f+1} RMSE:",{targets[k]:round(rmse[k],3) for k in range(4)})
    return oof_pred,oof_std,oof_mask,np.array(fold_rmse)

print("ready")

ready


## Run the cross-fitting training

This trains the three fold models on their full training sets and collects the out-of-fold predictions. The per-fold RMSE is reported as each fold finishes.

In [6]:
t0=time.time()
oof_pred,oof_std,oof_mask,fold_rmse=run_crossfit()
mean_rmse=fold_rmse.mean(axis=0)
std_rmse=fold_rmse.std(axis=0)
print()
print("MT-TrajNet RMSE per target (mean, fold spread) — stratified folds, cross-fitting:")
for k in range(4):
    print(f"  {targets[k]}: {mean_rmse[k]:.3f} (std {std_rmse[k]:.3f})  folds {[round(f,3) for f in fold_rmse[:,k]]}")
print("\ncovered by out-of-fold predictions:",int(oof_mask.sum()),"of",len(X_traj),"(code 15 excluded)")
print("time:",round(time.time()-t0),"seconds")

fold 1 RMSE: {'dissolution_av': np.float64(3.344), 'tbl_av_hardness': np.float64(7.9), 'tbl_rsd_weight': np.float64(0.77), 'fct_tensile': np.float64(0.404)}
fold 2 RMSE: {'dissolution_av': np.float64(3.147), 'tbl_av_hardness': np.float64(7.428), 'tbl_rsd_weight': np.float64(0.442), 'fct_tensile': np.float64(0.35)}
fold 3 RMSE: {'dissolution_av': np.float64(3.292), 'tbl_av_hardness': np.float64(7.767), 'tbl_rsd_weight': np.float64(0.543), 'fct_tensile': np.float64(0.283)}

MT-TrajNet RMSE per target (mean, fold spread) — stratified folds, cross-fitting:
  dissolution_av: 3.261 (std 0.083)  folds [np.float64(3.344), np.float64(3.147), np.float64(3.292)]
  tbl_av_hardness: 7.698 (std 0.198)  folds [np.float64(7.9), np.float64(7.428), np.float64(7.767)]
  tbl_rsd_weight: 0.585 (std 0.137)  folds [np.float64(0.77), np.float64(0.442), np.float64(0.543)]
  fct_tensile: 0.345 (std 0.050)  folds [np.float64(0.404), np.float64(0.35), np.float64(0.283)]

covered by out-of-fold predictions: 941 of

## Calibration from the out-of-fold predictions

The calibration scale for each target is estimated from the pooled out-of-fold predictions and their errors, then applied to the same predictions to measure the coverage and the expected calibration error. Because every prediction comes from a model that never saw that batch in training, this estimate does not leak.

In [7]:
z90=norm.ppf(0.95)

def compute_ece(y,pred,std,n_bins=10):
    z=np.abs(y-pred)/std
    levels=np.linspace(0.05,0.95,n_bins)
    e=0.0
    for c in levels:
        zc=norm.ppf(0.5+c/2)
        e+=abs(np.mean(z<=zc)-c)
    return e/len(levels)

m=oof_mask
print(f"{'target':<18}{'PICP(90%)':<12}{'ECE':<10}{'unc-err corr':<14}{'scale'}")
cal={}
for k in range(4):
    p=oof_pred[m,k];t=y_arr[m,k];s=oof_std[m,k]
    scale=np.sqrt(np.mean((p-t)**2))/np.mean(s)
    s_cal=s*scale
    picp=np.mean((t>=p-z90*s_cal)&(t<=p+z90*s_cal))
    ece=compute_ece(t,p,s_cal)
    corr=np.corrcoef(s_cal,np.abs(p-t))[0,1]
    cal[targets[k]]={"picp":round(float(picp),3),"ece":round(float(ece),3),"corr":round(float(corr),3),"scale":round(float(scale),3)}
    print(f"{targets[k]:<18}{picp:<12.3f}{ece:<10.3f}{corr:<14.3f}{scale:.3f}")

print("\nPICP near 0.90 = well calibrated. ECE near 0 = well calibrated.")

target            PICP(90%)   ECE       unc-err corr  scale
dissolution_av    0.858       0.022     0.039         0.211
tbl_av_hardness   0.933       0.038     0.531         0.316
tbl_rsd_weight    0.948       0.184     0.031         0.574
fct_tensile       0.850       0.025     0.014         0.253

PICP near 0.90 = well calibrated. ECE near 0 = well calibrated.


## Save the authoritative results

Everything from this single seeded run goes into one file: the per-fold RMSE, the mean and spread, and the calibration from cross-fitting. This is the authoritative result, replacing the earlier runs.

In [8]:
import json

final={
"run":"MT-TrajNet final, stratified folds + cross-fitting, seed 42",
"seed":SEED,
"setup":"3-fold stratified GroupKFold (high-cluster balanced 22/6/6), code 15 held out for OOD, cross-fitting calibration, stride-2, MAXLEN 6000, hidden 128",
"ood_code":OOD_CODE,
"oof_batches":int(oof_mask.sum()),
"rmse_mean":{targets[k]:round(float(mean_rmse[k]),3) for k in range(4)},
"rmse_std":{targets[k]:round(float(std_rmse[k]),3) for k in range(4)},
"rmse_folds":{targets[k]:[round(float(f),3) for f in fold_rmse[:,k]] for k in range(4)},
"calibration":cal,
"note":"stratified folds removed the fold-1 hardness artifact (was 19.7, now 7.4-7.9); cross-fitting recovered training data so RMSE improved; MT-TrajNet now beats the dummy on dissolution, hardness and tensile but not weight RSD; hardness still trails tuned XGBoost (7.7 vs 4.8); weight RSD uncertainty is poorly calibrated (ECE 0.184, correlation 0.03)"
}
with open("/kaggle/working/mttrajnet_stratified.json","w") as fh:
    json.dump(final,fh,indent=2)
print(json.dumps(final,indent=2))

{
  "run": "MT-TrajNet final, stratified folds + cross-fitting, seed 42",
  "seed": 42,
  "setup": "3-fold stratified GroupKFold (high-cluster balanced 22/6/6), code 15 held out for OOD, cross-fitting calibration, stride-2, MAXLEN 6000, hidden 128",
  "ood_code": 15,
  "oof_batches": 941,
  "rmse_mean": {
    "dissolution_av": 3.261,
    "tbl_av_hardness": 7.698,
    "tbl_rsd_weight": 0.585,
    "fct_tensile": 0.345
  },
  "rmse_std": {
    "dissolution_av": 0.083,
    "tbl_av_hardness": 0.198,
    "tbl_rsd_weight": 0.137,
    "fct_tensile": 0.05
  },
  "rmse_folds": {
    "dissolution_av": [
      3.344,
      3.147,
      3.292
    ],
    "tbl_av_hardness": [
      7.9,
      7.428,
      7.767
    ],
    "tbl_rsd_weight": [
      0.77,
      0.442,
      0.543
    ],
    "fct_tensile": [
      0.404,
      0.35,
      0.283
    ]
  },
  "calibration": {
    "dissolution_av": {
      "picp": 0.858,
      "ece": 0.022,
      "corr": 0.039,
      "scale": 0.211
    },
    "tbl_av_hardn

### Correction to the note field

The note printed by the cell above quotes the XGBoost hardness figure from the earlier pre-stratified baselines (5.0). That number does not apply to this run. On the stratified folds with code 15 held out, the tuned XGBoost hardness RMSE is 4.761, computed in Notebook 9 on these same folds. The note string in the cell source and the saved mttrajnet_stratified.json both carry the corrected figure of 4.8.

The cell output below the code is left exactly as the run produced it. Nothing else in the note changed and no model result in this run is affected: the RMSE, the fold values and the calibration numbers are all unchanged. The notebook was not re-run to make this correction, because re-running would retrain the authoritative model to fix a sentence.

## Save one trained model for SHAP and the OOD test

The cross-fitting run trains a model per fold and discards it. SHAP attribution and the out-of-distribution test both need a saved model to work from, so here I train one model on the first fold's training split, using the same seed, architecture and early stopping as the main run, and save it. I also save the normalisation statistics and the fold-1 train and test indices, so the attribution can prepare its inputs exactly as the model saw them during training.

In [9]:
te=fold_test_idx[0]
trv=np.concatenate([fold_test_idx[j] for j in range(n_splits) if j!=0])
g=groups[trv];uniq=np.unique(g)
rng=np.random.RandomState(SEED)
val_codes=rng.choice(uniq,size=max(1,len(uniq)//8),replace=False)
va=trv[np.isin(g,val_codes)]
tr=trv[~np.isin(g,val_codes)]

xmean=np.vstack([X_traj[i][::STRIDE] for i in tr]).mean(axis=0)
xstd=np.vstack([X_traj[i][::STRIDE] for i in tr]).std(axis=0)+1e-6
Xtr=prep_batch([X_traj[i] for i in tr],xmean,xstd)
Xva=prep_batch([X_traj[i] for i in va],xmean,xstd)
ymean=y_arr[tr].mean(axis=0);ystd=y_arr[tr].std(axis=0)+1e-6
ytr=torch.tensor((y_arr[tr]-ymean)/ystd,dtype=torch.float32)
yva=torch.tensor((y_arr[va]-ymean)/ystd,dtype=torch.float32)

torch.manual_seed(SEED)
shap_model=MTTrajNet().to(device)
opt=torch.optim.Adam(shap_model.parameters(),lr=5e-4)
best=float("inf");best_state=None;wait=0
for ep in range(150):
    shap_model.train()
    perm=torch.randperm(len(Xtr))
    for i in range(0,len(Xtr),16):
        idx=perm[i:i+16]
        xb=Xtr[idx].to(device);yb=ytr[idx].to(device)
        opt.zero_grad()
        gg,nn_,aa,bb=shap_model(xb)
        loss=sum(evidential_loss(yb[:,k],gg[:,k],nn_[:,k],aa[:,k],bb[:,k]) for k in range(4))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(shap_model.parameters(),1.0)
        opt.step()
    shap_model.eval()
    with torch.no_grad():
        vl=0
        for i in range(0,len(Xva),16):
            xb=Xva[i:i+16].to(device);yb=yva[i:i+16].to(device)
            gg,nn_,aa,bb=shap_model(xb)
            vl+=sum(evidential_loss(yb[:,k],gg[:,k],nn_[:,k],aa[:,k],bb[:,k]) for k in range(4)).item()
    if vl<best-1e-4:
        best=vl;best_state={kk:v.cpu().clone() for kk,v in shap_model.state_dict().items()};wait=0
    else:
        wait+=1
        if wait>=10:break
shap_model.load_state_dict(best_state)
torch.save(shap_model.state_dict(),"/kaggle/working/mttrajnet_stratified_fold1.pt")
np.save("/kaggle/working/shap_norm.npy",{"xmean":xmean,"xstd":xstd,"ymean":ymean,"ystd":ystd,"test_idx":te,"tr_idx":tr},allow_pickle=True)
print("model saved, stopped at epoch",ep+1,"| test batches",len(te))

model saved, stopped at epoch 32 | test batches 314
